<a href="https://colab.research.google.com/github/Fisev/PZP-Project/blob/Petlu/Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import re
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np



#get files
!wget https://raw.githubusercontent.com/Fisev/PZP-Project/refs/heads/main/data.txt -O data.txt
!wget https://raw.githubusercontent.com/Fisev/PZP-Project/refs/heads/main/stop_words.txt -O stop_words.txt
!pip install pycuda
pattern = re.compile(r'[^a-zA-Z]')
min_length = 4
max_length = 8

--2024-11-25 17:37:38--  https://raw.githubusercontent.com/Fisev/PZP-Project/refs/heads/main/data.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1257260 (1.2M) [text/plain]
Saving to: ‘data.txt’

data.txt            100%[===================>]   1.20M  --.-KB/s    in 0.09s   

2024-11-25 17:37:38 (13.4 MB/s) - ‘data.txt’ saved [1257260/1257260]

--2024-11-25 17:37:38--  https://raw.githubusercontent.com/Fisev/PZP-Project/refs/heads/main/stop_words.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 100 [text/plain]
Saving to: 

CPU SINGLE TREADED

In [7]:
try:
    # Read lullaby text
    with open("data.txt", "r") as file:
        f_lullaby = file.read().lower()  # Read the entire text as a string

    # Read stop words and create a set for quick lookups
    with open("stop_words.txt", "r") as file:
        f_stopW = set(file.read().lower().split())  # Use a set for faster membership checking

    # Clean the text: remove punctuation
    cleaned_text = pattern.sub(' ', f_lullaby)

    # Split the cleaned text into words
    words = cleaned_text.split()

    # Create a Counter to track word occurrences directly while filtering
    word_counts = Counter()
    filtered_words = []  # List to store filtered words

    # Filter and count words in a single loop
    for word in words:
        cleaned_word = pattern.sub('', word)  # Clean each word
        if min_length <= len(cleaned_word) <= max_length and cleaned_word not in f_stopW:
            word_counts[cleaned_word] += 1
            filtered_words.append(cleaned_word)  # Store the filtered word

    # Get the most and least frequent words
    most_frequent_word, most_frequent_count = word_counts.most_common(1)[0] if word_counts else (None, 0)
    least_frequent_word, least_frequent_count = min(word_counts.items(), key=lambda x: x[1], default=(None, 0))

    # Total number of words after filtering
    total_filtered_words = sum(word_counts.values())

    # Print the results
    print(f"Stop words: {list(f_stopW)}")  # Optionally print stop words for reference
    print(f"Filtered words: {filtered_words}")  # Print the filtered words
    print(f"Most frequent word: '{most_frequent_word}' with {most_frequent_count} occurrences")
    print(f"Least frequent word: '{least_frequent_word}' with {least_frequent_count} occurrences")
    print(f"Total number of filtered words: {total_filtered_words}")

except Exception as e:
    print(f"Reading failed: {e}")

Stop words: ['queequeg', 'ferrule', 'version', 'barbarians', 'warranty', 'gutenberg', 'summer-house', 'thee', 'electronic', 'odorous']
Filtered words: ['project', 'ebook', 'moby', 'dick', 'whale', 'herman', 'melville', 'this', 'ebook', 'anyone', 'anywhere', 'cost', 'with', 'almost', 'copy', 'give', 'away', 'under', 'terms', 'project', 'license', 'included', 'with', 'this', 'ebook', 'online', 'title', 'moby', 'dick', 'whale', 'author', 'herman', 'melville', 'last', 'updated', 'january', 'posting', 'date', 'december', 'ebook', 'release', 'date', 'june', 'language', 'english', 'start', 'this', 'project', 'ebook', 'moby', 'dick', 'whale', 'produced', 'daniel', 'lazarus', 'jonesey', 'moby', 'dick', 'whale', 'herman', 'melville', 'original', 'notes', 'this', 'text', 'etexts', 'from', 'defunct', 'eris', 'project', 'virginia', 'tech', 'from', 'project', 'archives', 'this', 'indebted', 'adelaide', 'library', 'virginia', 'tech', 'etext', 'compared', 'with', 'public', 'domain', 'hard', 'copy', 't

In [12]:
from pycuda.compiler import SourceModule
import pycuda.autoinit
import pycuda.driver as cuda
byte_array_text = np.array(list(f_lullaby.encode('utf-8')) + [0], dtype=np.uint8)
stop_words_bytes = b'\0'.join([word.encode('utf-8') for word in f_stopW]) + b'\0'

# Příprava GPU paměti
text_gpu = cuda.mem_alloc(byte_array_text.nbytes)
stop_words_gpu = cuda.mem_alloc(len(stop_words_bytes))
output_gpu = cuda.mem_alloc(byte_array_text.nbytes)

cuda.memcpy_htod(text_gpu, byte_array_text)
cuda.memcpy_htod(stop_words_gpu, stop_words_bytes)

# CUDA kód pro filtrování stop slov
gpu_code = '''
__global__ void filter_text(const unsigned char *text, const unsigned char *stop_words, unsigned char *output) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;
    int word_start = idx;  // Začátek aktuálního slova
    int output_pos = 0;    // Výstupní pozice

    // Procházení textu
    while (text[idx] != 0) {
        // Kontrola konce slova
        if (text[idx] == ' ' || text[idx] == 0) {
            int word_length = idx - word_start;
            bool is_stop_word = false;

            // Kontrola, zda je slovo ve stop-slovech
            for (int sw = 0; stop_words[sw] != 0; sw++) {
                if (strncmp((char *)&text[word_start], (char *)&stop_words[sw], word_length) == 0) {
                    is_stop_word = true;
                    break;
                }
            }

            // Pokud není stop slovo, zkopírujeme ho do výstupu
            if (!is_stop_word) {
                for (int i = word_start; i < idx; i++) {
                    output[output_pos++] = text[i];
                }
                output[output_pos++] = ' ';
            }

            // Posun na další slovo
            word_start = idx + 1;
        }
        idx++;
    }

    // Označení konce výstupu
    output[output_pos] = 0;
}
'''

mod = SourceModule(gpu_code)
filter_text = mod.get_function("filter_text")

# Spuštění CUDA jádra
block_size = 256
grid_size = (len(byte_array_text) + block_size - 1) // block_size

filter_text(cuda.In(byte_array_text), cuda.In(stop_words_bytes), output_gpu,
            block=(block_size, 1, 1), grid=(grid_size, 1, 1))

# Načtení výsledků z GPU
filtered_text_bytes = np.empty_like(byte_array_text)
cuda.memcpy_dtoh(filtered_text_bytes, output_gpu)

# Převod zpět na text
filtered_text = filtered_text_bytes.tobytes().decode('utf-8').strip()

print("Původní text:", whole_text)
print("Filtrovaný text:", filtered_text)


ImportError: libcuda.so.1: cannot open shared object file: No such file or directory